In [1]:
from Objects.Transformations import *
from Objects.WSBM import *
from Objects.TWSBMInstance import *

from Computation.Computation import *
from Computation.ExtraMetrics import *

from Plotting.Plotting import *
from Plotting.ArtisticPlotting import *
import cloudpickle

emb_mode = 'sqrt-scaled'
p22 = 'fixed'
n_batch = 5
eps = False

In [ ]:
import os
import numpy as np

def merge_dico(file_dir, prefixes, n_batch):
	for b in range(n_batch):
		b_path = os.path.join(file_dir, f"{b}.npz")
		if os.path.exists(b_path):
			merged = dict(np.load(b_path))
			######
			normalized = {}
			for key, val in merged.items():
				id_part, *rest = key.split('_', 1)
				if id_part.startswith('Pow-'):
					# parse the numeric part, then format to two decimals
					num = float(id_part.split('-', 1)[1])
					new_id = f'P-{num:.2f}'
					# rebuild the key
					suffix = rest[0] if rest else ''
					normalized_key = f'{new_id}_{suffix}' if suffix else new_id
				else:
					normalized_key = key
				normalized[normalized_key] = val
			merged = normalized
			######
		else:
			merged = {}
		for p in prefixes:
			p_path = os.path.join(file_dir, f"{p}{b}.npz")
			if os.path.exists(p_path):
				merged.update(dict(np.load(p_path)))
				os.remove(p_path)
		np.savez_compressed(b_path, **merged)

prefixes = []
path     = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}"
for rho, pi, model in RHOS_PIS_MODELS:
	file_dir = f"{path}/{model.__name__}_{rho}_{pi}".replace(".", "")
	merge_dico(file_dir, prefixes, n_batch)

In [ ]:
shifts  = np.concatenate([np.linspace(-15, -5, 20, endpoint=False),
					  np.linspace(-5, 5, 41, endpoint=True)])
windows = np.concatenate([np.linspace(0, 1.25, 20, endpoint=False), 
					  np.linspace(1.25, 2.5, 10, endpoint=False), 
					  np.linspace(2.5, 5, 11, endpoint=True)])


best_rand_avg = {model: {C: {} for C in GATED_CHERNOFFS_ID} for model in MODELS}

for model in MODELS:
	base = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}/{model.__name__}".replace('.', '')
	R = len(RHOS) * len(PIS)
	B = n_batch
	T = len(TRANSFORMS_EXT)

	grids = [[np.load(f"{base}_{rho}_{pi}/{b}".replace('.', '') + '.npz') for b in range(B)] for rho, pi in product(RHOS, PIS)]
	rand_shape = grids[0][0][f"{TRANSFORMS_EXT[0].id}_Rand"].shape
	N, M = rand_shape[0], rand_shape[1]
	assert N == M, "The Rand shape is not square"

	rand_stack = np.stack([g[f"{t.id}_Rand"] for batch in grids for t in TRANSFORMS_EXT for g in batch], axis=0).reshape(R, T, B, N, N)
	rand_mean = rand_stack.mean(axis=2).transpose(0,2,3,1)  # (R, N, N, T)

	gmm_stack = np.stack([g[f"{t.id}_GMM_score"] for batch in grids for t in TRANSFORMS_EXT for g in batch], axis=0).reshape(R, T, B, N, N)
	gmm_mean = gmm_stack.mean(axis=2).transpose(0,2,3,1)    # (R, N, N, T)

	C_count = len(GATED_CHERNOFFS_ID)
	metrics = np.stack(
		[g[f"{t.id}_{C[1:]}"]
		 for C in GATED_CHERNOFFS_ID
		 for t in TRANSFORMS_EXT
		 for batch in grids
		 for g in batch],
		axis=0
	).reshape(C_count, T, R, B, N, N)
	metrics = metrics.mean(axis=3).transpose(0,2,3,4,1)  # (C_count, R, N, N, T)
	best_rand_avg[model]["max"] = (rand_mean.max(axis=-1).mean())

	for c_idx, C in enumerate(GATED_CHERNOFFS_ID):
		M0 = metrics[c_idx]  # (R, N, N, T)
		Arg0 = np.argmax(M0, axis=-1)  # (R, N, N)
		best_rand_avg[model][C]["no gating"] = (np.take_along_axis(rand_mean, Arg0[..., None], axis=-1).mean())

		for x0, w in product(shifts, windows):
			gate = sigmoid_w95(gmm_mean, x0, w)  # (R, N, N, T)
			weighted = M0 * gate
			Arg_hp = np.argmax(weighted, axis=-1)  # (R, N, N)
			best_rand_avg[model][C][(x0, w)] = (np.take_along_axis(rand_mean, Arg_hp[..., None], axis=-1).mean())

with open('Computation/best_rand_avg.cpkl', 'wb') as f:
	cloudpickle.dump(best_rand_avg, f)

In [ ]:
path = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}"
metrics_g = {}
for rho, pi, model in RHOS_PIS_MODELS:
	file = f"{path}/{model.__name__}_{rho}_{pi}".replace(".", "")
	grids_stacked = [np.load(f"{file}/{b}.npz") for b in range(n_batch)]
	metrics_g[(rho, pi, model)] = {}
	for t in TRANSFORMS_EXT:
		metrics_g[(rho, pi, model)][t] = {}
		metrics_g[(rho, pi, model)][t]['std'] = {}
		for metric in VANILLA_METRICS_ID:
			g_stack = np.concatenate([g[f'{t.id}_{metric}'] for g in grids_stacked], axis = -1)
			mean = np.mean(g_stack, axis = -1)
			std  = np.std(g_stack, axis = -1)
			metrics_g[(rho, pi, model)][t][metric] = mean
			metrics_g[(rho, pi, model)][t]['std'][metric] = std
		for metric in GATED_CHERNOFFS_ID:
			gating_f = GATING_FUNCTIONS[metric]
			g_stack = np.concatenate([g[f'{t.id}_{metric[1:]}'] * gating_f(g[f'{t.id}_GMM_score'])
							 for g in grids_stacked], axis = -1)
			mean = np.mean(g_stack, axis = -1)
			std  = np.std(g_stack, axis = -1)
			metrics_g[(rho, pi, model)][t][metric] = mean
			metrics_g[(rho, pi, model)][t]['std'][metric] = std

metrics_g = aggregate_metrics(metrics_g)
metrics_g = best_transform_metrics(metrics_g)

for model in MODELS:
	metrics_g[model] = best_transform_metrics(metrics_g[model])

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	metrics_g[(rho, pi, model)] = best_transform_metrics(m)
	for t in TRANSFORMS_EXT:
		m = metrics_g[(rho, pi, model)][t]
		metrics_g[(rho, pi, model)][t] = correlation(m)
		metrics_g[(rho, pi, model)][t] = bias(m)

for model in MODELS:
	metrics_g[model] = correlation(metrics_g[model], light=True)
	for t in TRANSFORMS_EXT:
		metrics_g[model][t] = correlation(metrics_g[model][t], light=True)
	for rho in RHOS:
		metrics_g[model][f'rho:{rho}'] = correlation(metrics_g[model][f'rho:{rho}'], light=True)
	for pi in PIS:
		metrics_g[model][f'pi:{pi}'] = correlation(metrics_g[model][f'pi:{pi}'], light=True)
	for rho, pi, t in product(RHOS, PIS, TRANSFORMS_EXT):
		metrics_g[model][(rho, pi, t)] = correlation(metrics_g[model][(rho, pi, t)], light=True)

with open('Computation/metrics_g.cpkl', 'wb') as f:
	cloudpickle.dump(metrics_g, f)
plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\Documents\EPFL\MA6\Project\Code\Computation\ExtraMetrics.py:85: RuntimeWarning: divide by zero encountered in log
  return np.log(pred / (true + eps))
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\Documents\EPFL\MA6\Project\Code\Computation

In [2]:
with open('Computation/metrics_g.cpkl', 'rb') as f:
	metrics_g = cloudpickle.load(f)
with open('Computation/best_rand_avg.cpkl', 'rb') as f:
	best_rand_avg = cloudpickle.load(f)
plotter = Plotter(folder_path="", eps = eps)

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	plotter.plot_best_transform_heatmaps(rho, pi, model, m, gated = False)
	plotter.plot_best_transform_heatmaps(rho, pi, model, m, gated = True)
	plotter.plot_best_transform_scatter_rand_vs_regretratio(rho, pi, model, m)

In [4]:
# TODO : std (en vérité moyenne des stds)

for model in MODELS:
	plotter.plot_best_transform_scatter_rand_vs_regretratio(None, None, model, metrics_g[model])
	for mode in ['No regret', 'With regret']:
		for transforms, name in zip([TRANSFORMS, TRANSFORMS_POW, TRANSFORMS_QTL], 
						['Transforms', 'Powers', 'Quantiles']):
			plotter.plot_transforms_rand(model, metrics_g[model], mode = mode, 
								transforms = transforms.copy(), chernoffs = [], name = name)
		plotter.plot_transforms_rand(model, metrics_g[model], mode = mode, 
							transforms = TRANSFORMS_QTL[2:3], chernoffs = CHERNOFFS_ID.copy(), name = "Chernoffs")

	for chernoff in CHERNOFFS_ID:
		plotter.plot_transforms_rand_for_best_transform(model, metrics_g[model], chernoff)

In [5]:
for C in GATED_CHERNOFFS_ID:
	plotter.plot_rand_by_sigmoid_params_model_wise(C, best_rand_avg)
	plotter.plot_rand_by_sigmoid_params(C, best_rand_avg)

In [ ]:
for model in MODELS:
	for m_id2 in METRICS_ID[1:]:
		plotter.plot_correlation(model, metrics_g, 'Rand', m_id2, 'Reds')
	for m_id2 in METRICS_ID[2:]:
		plotter.plot_correlation(model, metrics_g, 'GMM_score', m_id2, 'Greens')
	for m_id2 in NON_GATED_CHERNOFFS_ID[1:]:
		plotter.plot_correlation(model, metrics_g, 'C_true', m_id2, 'Blues')
	for m_id2 in GATED_CHERNOFFS_ID[1:]:
		plotter.plot_correlation(model, metrics_g, 'gC_true', m_id2, 'Blues')

In [ ]:
"""
for rho, pi, model in RHOS_PIS_MODELS:
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)

for rho, pi, model in RHOS_PIS_MODELS:
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_bias_heatmap(rho, pi, model, t, m, log = True)"""

#corr(R,GMM), corr(C,GMM) (option verbose corr / simple corr)

rho, pi, model, t = 0.25, 0.1, betaWSBM, QuantileTransform(q=0.01)
m = metrics_g[(rho, pi, model)][t]
plotter.plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)
#plotter.plot_bias_heatmap(rho, pi, model, t, m, log = True)